# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

> **Note on keys:** this lab needs a **Pinecone** API key (free tier at [app.pinecone.io](https://app.pinecone.io/)). The setup cell below reads it securely from Google Colab's Secrets manager — no key typed in. `OPENAI_API_KEY` is loaded the same way for consistency with the other labs, but note the original notebook never actually uses it anywhere in the code below (it's dead code left over from a template) — everything here runs on Pinecone + a local audio embedding model instead.

# Install Dependencies

In [ ]:
#!pip install librosa
#!pip install panns-inference

In [ ]:
!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [ ]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})


The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [ ]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

[{'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.]),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.01184082, -0.10336304, -0.14141846, ...,  0.06985474,
          0.04049683,  0.00274658]),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.00695801, -0.01251221, -0.01126099, ...,  0.215271  ,
         -0.00875854, -0.28903198]),
  'sampling_rate': 44100}]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [ ]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [ ]:
from panns_inference import AudioTagging
import torch

# Auto-detect GPU vs CPU instead of hardcoding device='mps' (Apple Silicon only,
# won't exist on a Colab runtime). Uses a Colab/CUDA GPU automatically if present.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# load the default model
model = AudioTagging(checkpoint_path=None, device=device)

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [ ]:
# Load the Pinecone (and OpenAI, unused but kept for consistency) API keys
# securely from Google Colab's Secrets manager. In Colab: click the key icon in
# the left sidebar, add a secret named "PINECONE_API_KEY" (get one free at
# https://app.pinecone.io/) and, optionally, "OPENAI_API_KEY", then toggle
# "Notebook access" ON for THIS notebook for each. No key is ever typed here.
PINECONE_API_KEY = None
OPENAI_API_KEY = None
_source = None

try:
    from google.colab import userdata
    try:
        PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
        try:
            OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')  # optional, unused below
        except Exception:
            pass
        _source = "Colab secret"
    except Exception as e:
        print(f"Could not read the Colab secret PINECONE_API_KEY: {e}")
        print("Fix: open the key icon in the left sidebar, confirm a secret named")
        print("exactly PINECONE_API_KEY exists, and toggle Notebook access ON for this notebook.")
except ImportError:
    # Not running in Colab - fall back to a local .env file.
    import os
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv())
    PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    _source = ".env file" if PINECONE_API_KEY else None

assert PINECONE_API_KEY, (
    "PINECONE_API_KEY not found. In Colab: add it via the Secrets manager "
    "(key icon in the left sidebar) and enable notebook access for THIS notebook. "
    "Outside Colab: put PINECONE_API_KEY=... in a .env file next to this notebook."
)
print(f"PINECONE_API_KEY loaded successfully from {_source}.")

In [ ]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [ ]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [ ]:
index_name = "audio-search-demo"

In [ ]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [ ]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [ ]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [ ]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [ ]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '400', 'score': 0.999442935, 'values': []},
             {'id': '1667', 'score': 0.84225893, 'values': []},
             {'id': '1666', 'score': 0.831919551, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 5}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [ ]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [ ]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [ ]:
find_similar_audios(1642)

Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [ ]:
find_similar_audios(452)

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [ ]:
import os
from urllib.request import urlretrieve

wav_path = "miaow_16k.wav"
if not os.path.exists(wav_path):
    # If you already have miaow_16k.wav attached to this lab, just place it in
    # the notebook's working directory and this download will be skipped.
    urlretrieve("https://storage.googleapis.com/audioset/miaow_16k.wav", wav_path)
    print(f"Downloaded {wav_path}")
else:
    print(f"Using existing {wav_path}")

We can load the audio into a Numpy array as follows:

In [ ]:
import librosa

a, _ = librosa.load(wav_path, sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [ ]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [ ]:
pc.delete_index(index_name)

---
## My Solution: A Local RAG Music Recommender Built on a Real Music Dataset

The demo above is a great template, but it's built on **ESC-50** — environmental sounds (car horns, birds, lawnmowers), not music. Per the lab's instructions, I built a second, independent pipeline using an actual **music** dataset with genre metadata, reusing the exact same PANNs embedding model + Pinecone pattern, and added a `recommend_songs` function that takes a **stated user preference** (a genre, in natural language) rather than requiring the user to already have a specific audio file/id in hand — closer to how a real recommendation feature would be used.

**Dataset:** [`lewtun/music_genres_small`](https://huggingface.co/datasets/lewtun/music_genres_small) — a small, easy-to-download Hugging Face audio dataset (used in the HF Audio Course) of short music clips labeled by genre (Electronic, Rock, Punk, Hip-Hop, Folk, Classical, Ambient, etc.). It's small enough to embed and index quickly, while still being real music with genre metadata to build a "preference" on top of.

**Why a separate index:** rather than mixing music into the `audio-search-demo` index (which is dimensioned/tuned for the ESC-50 environmental-sound demo above), I create a new Pinecone index, `music-recommendations-demo`, so the two catalogs don't collide.

In [ ]:
# Load a small real music dataset with genre labels.
from datasets import load_dataset

music_data = load_dataset("lewtun/music_genres_small", split="train")
music_data

In [ ]:
# Same pattern as the ESC-50 demo above: pull out the raw audio arrays and
# resample if needed so everything is consistent before embedding. This
# dataset's clips are already short enough to embed directly.
import numpy as np

music_audios = np.array([a["array"] for a in music_data["audio"]], dtype=object)
music_genres = music_data["genre"]

print(f"Loaded {len(music_audios)} tracks across genres: {sorted(set(music_genres))}")

In [ ]:
# A second, separate Pinecone index just for the music catalog, so it doesn't
# collide with the audio-search-demo index used for ESC-50 above.
music_index_name = "music-recommendations-demo"

if music_index_name not in pc.list_indexes().names():
    pc.create_index(
        music_index_name,
        dimension=2048,  # same PANNs embedding model, same 2048-dim output
        metric="cosine",
        spec=spec,
    )
    while not pc.describe_index(music_index_name).status["ready"]:
        time.sleep(1)

music_index = pc.Index(music_index_name)
music_index.describe_index_stats()

In [ ]:
# Embed and upsert the music catalog, storing each track's genre as metadata
# so we can later filter/seed recommendations by stated user preference.
batch_size = 32

for i in tqdm(range(0, len(music_audios), batch_size)):
    i_end = min(i + batch_size, len(music_audios))
    batch = list(music_audios[i:i_end])
    _, emb = model.inference(batch)
    ids = [f"{idx}" for idx in range(i, i_end)]
    metadata = [{"genre": music_genres[idx]} for idx in range(i, i_end)]
    to_upsert = list(zip(ids, emb.tolist(), metadata))
    _ = music_index.upsert(vectors=to_upsert)

music_index.describe_index_stats()

In [ ]:
def recommend_songs(preferred_genre: str, top_k: int = 5):
    """A simple preference-based RAG recommender:
    1. RETRIEVE a seed track whose genre metadata matches the user's stated
       preference (falls back to a case-insensitive partial match if there's
       no exact match).
    2. Use that seed track's audio embedding to search the vector index for
       the most sonically similar tracks - this is the actual "retrieval
       augmented" step: we don't just filter by the genre label, we search by
       audio similarity to find tracks that *sound* like what the user wants,
       which can surface good recommendations across adjacent/blended genres
       too, not just an exact genre match.
    3. Print out the recommended tracks (genre + similarity score) and play them.
    """
    # Step 1: find a seed track id matching the requested genre.
    seed_idx = next(
        (i for i, g in enumerate(music_genres) if preferred_genre.lower() in g.lower()),
        None,
    )
    if seed_idx is None:
        print(f"No tracks found matching genre '{preferred_genre}'. Available genres: {sorted(set(music_genres))}")
        return

    print(f"Seed track (genre: {music_genres[seed_idx]}):")
    display(Audio(music_audios[seed_idx], rate=music_data[seed_idx]["audio"]["sampling_rate"]))

    # Step 2: retrieve similar tracks by audio embedding, using the already-
    # indexed seed track's id (same pattern as find_similar_audios() above).
    result = music_index.query(id=str(seed_idx), top_k=top_k + 1)  # +1 since the seed itself will match

    print(f"\nRecommended tracks for someone who likes '{preferred_genre}':")
    for r in result["matches"]:
        rid = int(r["id"])
        if rid == seed_idx:
            continue  # skip the seed track itself
        print(f"  - Track {rid} | genre: {music_genres[rid]} | similarity: {r['score']:.3f}")
        display(Audio(music_audios[rid], rate=music_data[rid]["audio"]["sampling_rate"]))


# Try it out with a couple of different stated preferences.
recommend_songs("Electronic")
recommend_songs("Classical")

## Report: Findings from the Music Recommendation Extension

**What I built:** a second, independent RAG-style pipeline that mirrors the ESC-50 demo's architecture exactly (PANNs embeddings -> Pinecone vector index -> nearest-neighbor query) but applied to `lewtun/music_genres_small`, an actual small music dataset with genre labels, plus a `recommend_songs(preferred_genre)` function that takes a **natural-language preference** rather than requiring the caller to already know a specific track id.

**Why this counts as "RAG" for audio, not just similarity search:** the demo's `find_similar_audios(id)` requires you to already have a specific audio example in hand. Real recommendation ("play me something like the Electronic music I like") starts from a *stated preference*, not a specific file. `recommend_songs` bridges that gap in two retrieval steps: first retrieving a representative seed track for the stated genre (using metadata), then retrieving sonically-similar tracks to that seed via the audio embedding index (not just other tracks with the same genre label) — so the final recommendations are grounded in actual audio similarity, which can surface genre-adjacent tracks a pure metadata filter would miss, the same "retrieve broadly, then let similarity do the ranking" idea behind the retrieval-augmented pattern used in the text-based RAG labs earlier in this project.

**What I expect to find (to confirm once you run this with your own Pinecone key and Colab GPU/CPU):**
- Tracks within the same broad genre bucket (e.g., "Classical") should cluster tightly and return high similarity scores to each other, similar to how the ESC-50 demo's car-horn query scored its own audio at ~0.999 and other car-horn-like sounds close behind.
- Genres with more sonic variety internally (e.g., "Electronic," which spans everything from ambient to breakbeat) are the most interesting to watch — the top similarity matches may span sub-styles that don't share the exact genre label, which is precisely the case where embedding-based retrieval should outperform a plain genre filter.
- Because this dataset is much smaller than ESC-50 (a few dozen to a couple hundred tracks depending on the split, vs. ESC-50's 2000), don't expect very deep/varied playlists per genre from `top_k` alone — a production version would need a much larger music catalog (the lab's suggested Zenodo/GitHub datasets, or a larger Hugging Face audio dataset) to give a truly rich recommendation experience.

**What I learned:**
- The exact same three-part pattern (embed -> index -> nearest-neighbor query) generalizes cleanly from "find similar environmental sounds" to "recommend similar music" — nothing about the PANNs model or the Pinecone querying code needed to change, only the dataset and the *entry point* into the retrieval (a stated preference instead of a known audio id).
- Keeping the music catalog in its own Pinecone index (rather than mixing it into `audio-search-demo`) matters because ESC-50 and this music dataset are conceptually different catalogs — combining them into one index would make "similar sound" queries return, say, a barking dog next to a punk song, which isn't a meaningful recommendation even if PANNs embeds them nearby in some generic audio-features sense.
- A genre-label-seeded retrieval is a pragmatic middle ground between pure content-based filtering (genre tags only, no notion of *how* two songs actually sound) and pure embedding search with no starting point (which needs the user to already have an audio example) — this two-step "seed, then retrieve similar" approach is a reasonable minimal viable pattern for an audio recommender when you're starting from a stated preference in words rather than an uploaded song.